In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# RGB-only Baseline
# ResNet18 + Temporal Average Pooling
# ============================================

import os
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from tqdm import tqdm
from sklearn.metrics import accuracy_score


# ============================================
# Seed 고정
# ============================================

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)


# ============================================
# 경로 설정
# ============================================

DATA_ROOT = "/content/drive/MyDrive/salpim_project/02_processed_frames_person_split"

SAVE_PATH = "/content/drive/MyDrive/salpim_project/02_models/rgb_only_baseline_best.pth"


# ============================================
# 클래스 설정
# ============================================

CLASS_NAMES = [
    "A010",
    "A011",
    "A016",
    "A018",
    "A023",
    "A031",
    "A035",
    "A041",
    "A053",
    "A054"
]

label_map = {
    cls_name: idx
    for idx, cls_name in enumerate(CLASS_NAMES)
}

NUM_CLASSES = len(CLASS_NAMES)


# ============================================
# 하이퍼파라미터
# ============================================

IMG_SIZE = 224
NUM_FRAMES = 8

BATCH_SIZE = 8
NUM_WORKERS = 2

LR = 1e-4
EPOCHS = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)


# ============================================
# Transform
# ============================================

train_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================
# Frame Sampling
# ============================================

def sample_frames(frame_paths, num_frames=16):

    total_frames = len(frame_paths)

    # frame 수 부족하면 마지막 frame 반복
    if total_frames < num_frames:

        while len(frame_paths) < num_frames:

            frame_paths.append(frame_paths[-1])

    total_frames = len(frame_paths)

    # 균등 sampling
    indices = np.linspace(
        0,
        total_frames - 1,
        num_frames
    ).astype(int)

    sampled = [frame_paths[i] for i in indices]

    return sampled


# ============================================
# Dataset
# ============================================

class RGBVideoDataset(Dataset):

    def __init__(
        self,
        root_dir,
        split="train",
        transform=None,
        num_frames=16
    ):

        self.samples = []

        self.transform = transform

        self.num_frames = num_frames

        split_dir = os.path.join(root_dir, split)

        for action in os.listdir(split_dir):

            if action not in label_map:
                continue

            action_dir = os.path.join(split_dir, action)

            if not os.path.isdir(action_dir):
                continue

            for video_name in os.listdir(action_dir):

                video_dir = os.path.join(
                    action_dir,
                    video_name
                )

                if not os.path.isdir(video_dir):
                    continue

                self.samples.append({
                    "video_dir": video_dir,
                    "label": label_map[action]
                })

        print(f"{split} samples:", len(self.samples))

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        video_dir = sample["video_dir"]

        label = sample["label"]

        frame_files = sorted([
          os.path.join(video_dir, x)
          for x in os.listdir(video_dir)
          if ".jpg" in x.lower()
        ])

        if len(frame_files) == 0:

          print("NO FRAMES:", video_dir)

          print(os.listdir(video_dir))

          raise ValueError("Empty frame folder")#

        frame_files = sample_frames(
            frame_files,
            self.num_frames
        )

        frames = []

        for frame_path in frame_files:

            image = Image.open(frame_path).convert("RGB")

            if self.transform:
                image = self.transform(image)

            frames.append(image)

        frames = torch.stack(frames)

        return frames, label


# ============================================
# Dataloader
# ============================================

train_dataset = RGBVideoDataset(
    DATA_ROOT,
    split="train",
    transform=train_transform,
    num_frames=NUM_FRAMES
)

val_dataset = RGBVideoDataset(
    DATA_ROOT,
    split="val",
    transform=val_transform,
    num_frames=NUM_FRAMES
)

test_dataset = RGBVideoDataset(
    DATA_ROOT,
    split="test",
    transform=val_transform,
    num_frames=NUM_FRAMES
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# ============================================
# RGB-only Model
# ============================================

class RGBBaselineModel(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        backbone = models.resnet18(
            pretrained=True
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.classifier = nn.Sequential(

            nn.Dropout(0.3),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        # x:
        # (B, T, C, H, W)

        B, T, C, H, W = x.shape

        # frame 단위 처리
        x = x.view(B * T, C, H, W)

        feat = self.backbone(x)

        # (B*T, 512)
        feat = feat.view(B, T, 512)

        # temporal average pooling
        feat = feat.mean(dim=1)

        logits = self.classifier(feat)

        return logits


# ============================================
# Model
# ============================================

model = RGBBaselineModel(
    num_classes=NUM_CLASSES
).to(DEVICE)


# ============================================
# Loss / Optimizer
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)


# ============================================
# Train Function
# ============================================

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0

    preds_all = []
    labels_all = []

    for frames, labels in tqdm(loader):

        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(frames)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)

        preds_all.extend(
            preds.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    return total_loss / len(loader), acc


# ============================================
# Validation Function
# ============================================

def validate(model, loader):

    model.eval()

    total_loss = 0

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for frames, labels in tqdm(loader):

            frames = frames.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(frames)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    return total_loss / len(loader), acc


# ============================================
# Training
# ============================================

best_val_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = validate(
        model,
        val_loader
    )

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}]"
    )

    print(
        f"Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.4f}"
    )

    # best 저장
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            SAVE_PATH
        )

        print("Best model saved")


print("\nTraining Finished")
print("Best Val Acc:", best_val_acc)


# ============================================
# Test
# ============================================

print("\n===== TEST =====")

model.load_state_dict(
    torch.load(SAVE_PATH)
)

test_loss, test_acc = validate(
    model,
    test_loader
)

print(
    f"Test Loss: {test_loss:.4f}"
)

print(
    f"Test Acc: {test_acc:.4f}"
)

DEVICE: cuda
train samples: 629
val samples: 232
test samples: 171


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 151MB/s]
100%|██████████| 29/29 [04:22<00:00,  9.06s/it]



Epoch [1/10]
Train Loss: 1.8094 | Train Acc: 0.3434
Val Loss: 1.5641 | Val Acc: 0.4310
Best model saved


...


Epoch [9/10]
Train Loss: 0.0951 | Train Acc: 0.9841
Val Loss: 1.2086 | Val Acc: 0.6336
Best model saved


100%|██████████| 29/29 [00:46<00:00,  1.59s/it]



Epoch [10/10]
Train Loss: 0.0827 | Train Acc: 0.9777
Val Loss: 1.2355 | Val Acc: 0.6552
Best model saved

Training Finished
Best Val Acc: 0.6551724137931034

===== TEST =====


100%|██████████| 22/22 [03:15<00:00,  8.90s/it]

Test Loss: 1.0665
Test Acc: 0.6667
